# NovaTech AI — Week 1: EDA Notebook
> CRS AI Capstone 2025–26 · Scenario 1

Analyses all three datasets: sloganlist.csv, startups.csv, marketing_campaign_dataset.csv

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

from src.data_loader import load_slogans, load_startups, load_marketing
from src.preprocess  import clean_slogans, clean_startups, clean_marketing

df_slogans,  sl_real  = load_slogans()
df_startups, st_real  = load_startups()
df_marketing,mk_real  = load_marketing()

print(f'Slogans  : {len(df_slogans)} rows  (real={sl_real})')
print(f'Startups : {len(df_startups)} rows  (real={st_real})')
print(f'Marketing: {len(df_marketing)} rows (real={mk_real})')

## 1. Slogan EDA

In [ ]:
df_sl = clean_slogans(df_slogans)
print(df_sl.describe())
print('\nTop 20 companies:')
print(df_sl['Company'].value_counts().head(20))

In [ ]:
fig = px.histogram(df_sl, x='slogan_len', nbins=25,
    title='Slogan Word Count Distribution',
    color_discrete_sequence=['#C8A94A'],
    template='plotly_dark')
fig.show()

In [ ]:
# Top words in slogans
from collections import Counter
import re
stop = {'the','a','is','in','of','and','to','your','our','we','for','with','be','you'}
words = [w for sl in df_sl['Slogan'] for w in re.findall(r'\b[a-z]+\b', sl.lower()) if w not in stop and len(w)>2]
top_words = Counter(words).most_common(20)
df_words  = pd.DataFrame(top_words, columns=['word','count'])
fig = px.bar(df_words, x='count', y='word', orientation='h',
    title='Top 20 Words in Slogans', template='plotly_dark',
    color_discrete_sequence=['#C8A94A'])
fig.show()

## 2. Startup EDA

In [ ]:
df_st = clean_startups(df_startups)
print(df_st.head())
print('\nTop cities:')
print(df_st['city'].value_counts().head(10))

## 3. Marketing Campaign EDA

In [ ]:
df_mk = clean_marketing(df_marketing)
print(df_mk.describe())
print('\nNull counts:')
print(df_mk.isnull().sum())

In [ ]:
fig = px.box(df_mk, x='Campaign_Type', y='Engagement_Score',
    title='Engagement Score by Campaign Type',
    template='plotly_dark', color='Campaign_Type')
fig.show()

In [ ]:
roi_channel = df_mk.groupby('Channel_Used')['ROI'].mean().sort_values(ascending=False)
fig = px.bar(roi_channel.reset_index(), x='Channel_Used', y='ROI',
    title='Mean ROI by Channel', template='plotly_dark',
    color='ROI', color_continuous_scale='Oranges')
fig.show()

In [ ]:
fig = px.scatter(df_mk, x='Acquisition_Cost', y='ROI', color='Campaign_Type',
    title='Acquisition Cost vs ROI by Campaign Type',
    template='plotly_dark', opacity=0.6)
fig.show()

## 4. Feature Correlations

In [ ]:
num_cols = ['Duration','Acquisition_Cost','ROI','Clicks','Impressions','Engagement_Score','CTR']
num_cols = [c for c in num_cols if c in df_mk.columns]
corr = df_mk[num_cols].corr()
fig  = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index,
    colorscale='RdBu', zmid=0,
    text=corr.values.round(2), texttemplate='%{text}',
))
fig.update_layout(title='Feature Correlation Matrix', template='plotly_dark')
fig.show()